# 데이터 불러오기

In [ ]:
import pandas as pd
import numpy as np
import os
df_items = pd.read_csv("olist_order_items_dataset.csv")
df_reviews = pd.read_csv("olist_order_reviews_dataset.csv")
df_orders = pd.read_csv("olist_orders_dataset.csv")
df_products = pd.read_csv("olist_products_dataset.csv")
# df_geolocation = pd.read_csv('olist_geolocation_dataset.csv'))
df_sellers = pd.read_csv("olist_sellers_dataset.csv")
df_payments = pd.read_csv("olist_order_payments_dataset.csv")
df_customers = pd.read_csv("olist_customers_dataset.csv")
# df_category = pd.read_csv('product_category_name_translation.csv'))


FileNotFoundError: [Errno 2] No such file or directory: 'olist_order_items_dataset.csv'

# 데이터 합치기

In [ ]:
# 기본 주문 정보
df = df_orders
# 주문 아이템 정보 추가 (order_id 기준)
df = df.merge(df_items, on="order_id", how="outer")
# 결제 정보 추가 (order_id 기준, df_payments는 order_id 기준으로 다:다 가능하므로 validate='m:m')
df = df.merge(df_payments, on="order_id", how="outer")
# 리뷰 정보 추가 (order_id 기준)
df = df.merge(df_reviews, on="order_id", how="outer")
# 제품 정보 추가 (product_id 기준)
df = df.merge(df_products, on="product_id", how="outer")
# 고객 정보 추가 (customer_id 기준)
df = df.merge(df_customers, on="customer_id", how="outer")
# 판매자 정보 추가 (seller_id 기준)
df = df.merge(df_sellers, on="seller_id", how="outer")

# 최종 병합된 데이터셋 저장
# df.to_csv('merged_data.csv', encoding='utf-8')

# 결측치 처리 (Task 1)

In [ ]:
# 데이터 기본 정보 및 결측치 확인
print("데이터 형태(Shape):", df.shape)
print("\n--- 데이터 타입 및 결측치 개수 ---")
print(df.info())

In [ ]:
# 컬럼별 결측치 비율 확인
missing_percentage = df.isna().mean() * 100
print("\n--- 컬럼별 결측치 비율(%) ---")
print(missing_percentage[missing_percentage > 20].sort_values(ascending=False))

In [ ]:
# 결측치 비율 높은 (20% 이상) 열 제거
df = df.drop(columns=missing_percentage[missing_percentage >=20].index)

In [ ]:
# 나머지 열들 확인 결과, 모두 결측 비율 5% 미만이므로
# 결측치 모두 제거
df = df.dropna()

In [ ]:
# 확인
print(df.shape)
df.info()

# 이상치 처리

In [ ]:
# IQR
Q1 = df['price'].quantile(0.25)
Q3 = df['price'].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[(df['price'] < lower) | (df['price'] > upper)]

print(f"이상치: {len(outliers)} 개")
outliers[['price']].sort_values(by='price', ascending=False).head(10)

In [ ]:
# 화폐 단위 = 브라질 헤알(R$)로 추정
# 1 R$ = 약 270 \
# 6735 R$ = 1818450 \ -> 이커머스 상품 가격으로 비정상적이라고 판단되는지 ?

In [ ]:
# 시각화
import matplotlib.pyplot as plt

plt.figure()

# 박스플롯
plt.boxplot(df['price'], showfliers=True)

# 경계선 표시
plt.axhline(upper, linestyle='--', linewidth=2, label='Upper IQR Bound')
plt.axhline(lower, linestyle='--', linewidth=2, label='Lower IQR Bound')

plt.title('Boxplot of Price with IQR')
plt.ylabel('price')
plt.legend()
plt.show()

# 논리적 이상치 처리

In [ ]:
# 1. 배송사 출고일이 승인일보다 빠른 경우

# pd.datetime(X) -> X를 datetime 객체로 바꿔 반환함
# datetime 객체는 일, 월, 연 변환 및 비교가 용이하여 날짜 컬럼은 datetime으로 바꾸는 것이 좋음

# 필요 컬럼 두 가지 datetime 객체로 변환하기
df['order_delivered_carrier_date']= pd.to_datetime(df['order_delivered_carrier_date'])
df['order_approved_at']= pd.to_datetime(df['order_approved_at'])

invalid_df1 = df[df['order_delivered_carrier_date']< df['order_approved_at']]

print(invalid_df1.shape)
invalid_df1.head()

In [ ]:
# 2. 고객 수령일이 배송사 출발일보다 빠른 경우

# 필요 컬럼 두 가지 datetime 객체로 변환하기
df['order_delivered_customer_date']= pd.to_datetime(df['order_delivered_customer_date'])
df['order_delivered_carrier_date']= pd.to_datetime(df['order_delivered_carrier_date'])

invalid_df2 = df[df['order_delivered_customer_date'] < df['order_delivered_carrier_date']]

print(invalid_df2.shape)
invalid_df2.head()

In [ ]:
# 3. 리뷰 작성일이 고객 수령일보다 빠른 경우

# 필요 컬럼 두 가지 datetime 객체로 변환하기 (앞에서 했으면 안 해도 됨)
df['review_creation_date'] = pd.to_datetime(df['review_creation_date'])
df['order_delivered_customer_date'] = pd.to_datetime(df['order_delivered_customer_date'])

invalid_df3 = df[df['review_creation_date'] < df['order_delivered_customer_date']]

print(invalid_df3.shape)
invalid_df3.head()

In [ ]:
# 리뷰 작성일이 고객 수령일보다 빠른 것이 논리적 오류인가?
# 배달이 지연되는 경우, 수령하기 전에 구매 확정이 되어 리뷰 작성이 가능해졌을 가능성을 고려할 수 있음
# 판단은 자신의 몫 !

# 변수 변환

In [ ]:
price_df = df[["price"]].copy()

In [ ]:
# 표준화
# 평균을 0으로, 분산을 1로
from sklearn.preprocessing import StandardScaler

standard_scaler = StandardScaler()

price_df["price_standard"] = standard_scaler.fit_transform(price_df[["price"]])

In [ ]:
# 정규화
from sklearn.preprocessing import MinMaxScaler

minmax_scaler = MinMaxScaler()

price_df["price_minmax"] = minmax_scaler.fit_transform(price_df[["price"]])

In [ ]:
price_df

# 상관분석 해보기

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 시각화 스타일 설정
sns.set_palette('husl')
sns.set_style('whitegrid')

# 주문/배송 시각과 관련된 정보 불러오기
df_orders = pd.read_csv('olist_orders_dataset.csv')[[
    'order_id',
    'order_approved_at',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
]]
df = df_orders.copy()

# 날짜 계산을 위한 전처리
df['order_approved_at'] = pd.to_datetime(df['order_approved_at'])
df['order_delivered_customer_date'] = pd.to_datetime(
    df['order_delivered_customer_date'])
df['order_estimated_delivery_date'] = pd.to_datetime(
    df['order_estimated_delivery_date'])

# 계산된 값으로 새로운 칼럼(배송 지연일, 배송 소요일)을 추가
# 배송 지연일, 배송 소요일을 계산하려면 어떤 컬럼을 사용하면 될까요?
df['delivery_latency_days'] = (df['order_delivered_customer_date'] - df['order_approved_at']).dt.days
df['delivery_days'] = (df['order_delivered_customer_date'] - df['order_approved_at']).dt.days

# 주문 별 결제 금액 정보 가져오기
df_payments = pd.read_csv('olist_order_payments_dataset.csv')[[
    'order_id',
    'payment_value',
]]
df = pd.merge(left=df, right=df_payments, on='order_id', how='left')

# 주문 별 리뷰 점수 정보 가져오기
df_reviews = pd.read_csv('olist_order_reviews_dataset.csv')[[
    'order_id',
    'review_score',
]]
df = pd.merge(left=df, right=df_reviews, on='order_id', how='left')

# =========================================
# 한 주문에는 상품이 여러 개 있을 수 있다.
# =========================================

# 주문에 포함된 상품 별 배송비 & 무게 정보 가져오기
df_product = pd.merge(
    left=pd.read_csv('olist_order_items_dataset.csv')[[
        'order_id',
        'product_id',
        'freight_value',
    ]],
    right=pd.read_csv('olist_products_dataset.csv')[[
        'product_id',
        'product_weight_g',
    ]],
    on='product_id',
    how='left',
)

# 주문 별로 상품 배송비 & 무게 정보 집계하기
# 그룹화 함수, 집계 함수를 활용해보세요.
df_product_agg = df_product[['order_id', 'product_weight_g', 'freight_value']] \
    .groupby('order_id') \
    .agg({
        'product_weight_g': 'sum',
        'freight_value': 'sum',
    }) \
    .reset_index()
df = pd.merge(left=df, right=df_product_agg, on='order_id', how='left')


# 사용하지 않을 칼럼 제거
df.drop(
    columns=[
        'order_id',
        'order_approved_at',
        'order_delivered_customer_date',
        'order_estimated_delivery_date',
    ],
    inplace=True,
)

df.head(10)

In [ ]:
# 히트맵 그리기
import seaborn as sns
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)

In [ ]:
# 값이 0보다 멀수록 두 변수가 강하게 관계되어 있다
# 강한 연관: product_weight_g <-> freight_value
# 약한 연관: delivery_days <-> payments_value

# 추가 변수 창출

In [ ]:
df_items = pd.read_csv("olist_order_items_dataset.csv")
df_reviews = pd.read_csv("olist_order_reviews_dataset.csv")
df_orders = pd.read_csv("olist_orders_dataset.csv")
df_products = pd.read_csv("olist_products_dataset.csv")
df_sellers = pd.read_csv("olist_sellers_dataset.csv")
df_payments = pd.read_csv("olist_order_payments_dataset.csv")
df_customers = pd.read_csv("olist_customers_dataset.csv")

In [ ]:
# CLV(Customer Lifetime Value) 고객 생애 가치 구하기
# CLV = 평균 구매 금액 (AOV) X  평균 구매 빈도 (F) X 고객 유지 시간 (L)

# Average Order Value (AOV) 평균 구매 금액  총 구매 금액 / 구매 횟수
# Purchase Frequency (F) 평균 구매 빈도 구매 횟수 / 활동 기간
# Customer Lifespan (L) 고객이 유지되는 기간 예: 1년, 또는 고객별 실제 기간

# AOV를 계산하기 위해 merge 해주기
df_order = pd.merge(df_orders, df_payments, on='order_id', how='left')
df_order_with_customer = pd.merge(df_order, df_customers, on='customer_id', how='left')


# 주문 및 결제 데이터 불러오기
# 사용자 별 총 구매 금액 데이터프레임# 주문 및 결제 데이터 불러오기
# 사용자 별 총 구매 금액 데이터프레임을 total_payment에 저장하세요
# customer_unique_id 컬럼을 활용하세요
total_payment = df_order_with_customer.groupby('customer_unique_id')['payment_value'].sum()
# 사용자 별 총 주문 수 데이터프레임을 total_orders에 저장하세요
total_orders = df_order_with_customer.groupby('customer_unique_id')['order_id'].nunique()

# 평균 구매금액 (Average Order Value)
aov = total_payment/ total_orders

# 전체 기간(을 구하기 위해서는 str -> datetime 으로 형 변환을 해야한다)
timestamp = pd.to_datetime(df_order['order_purchase_timestamp'])
period = (timestamp.max() - timestamp.min()) # 데이터의 전체 기간
total_months = period.days / 30 # 기간의 개월 수

print(f'{period=}')

# 구매 빈도 (월 단위)
# Hint: 이미 구함
frequency = total_orders # 고객 별 period 내 구매 횟수

# 고객 생애 가치 계산 (고객의 수명은 6개월로 가정)
clv = aov * (frequency / total_months) * 6 # 빈도의 월 평균을 기준으로 6개월로 가정
print(clv)

In [ ]:
# dataframe 형태로 변환
clv = clv.reset_index()
clv.columns = ['customer_id', 'clv']

clv.head()

In [ ]:
# dataframe 형태로 변환
aov = aov.reset_index()
aov.columns = ['customer_id', 'aov']

aov.head()

In [ ]:
# dataframe 형태로 변환
frequency = frequency.reset_index()
frequency.columns = ['customer_id', 'frequency']

frequency.head()